<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/Megpie_RUL_Prediction_R04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import warnings

# 불필요한 경고창 숨기기
warnings.filterwarnings("ignore")

# ==========================================
# 0. Dummy Data Generation (가상 데이터 생성)
# ==========================================
# 현업의 sel_df 데이터가 있다고 가정하고 가상의 센서 데이터를 생성합니다.
# (실제 데이터 사용 시 이 부분은 삭제 또는 주석 처리 하세요)
def generate_dummy_data():
    np.random.seed(42)
    # 불규칙한 수집 주기를 묘사하기 위해 임의의 시간 데이터 생성 (1시간~3시간 간격 섞임)
    times = pd.date_range(start='2026-05-01', end='2026-05-30', freq='1h')
    times = times[np.random.rand(len(times)) > 0.3] # 약 30%의 데이터 누락 발생

    n = len(times)
    onset_idx = int(n * 0.5)

    # 기본 열화 베이스라인
    base_val = np.zeros(n)
    base_val[onset_idx:] = np.exp(np.linspace(0, 3.5, n - onset_idx)) / np.exp(3.5) * 1.3

    # 노이즈 및 웨이브 추가
    noise = np.random.normal(0, 0.03, n)
    wave = np.sin(np.linspace(0, 30, n)) * 0.08
    median_val = base_val + noise + wave

    # 이상치(Outlier) 및 에러(문자열 섞임) 추가
    median_val[int(n * 0.2)] = 0.9
    median_val[int(n * 0.7)] = 0.2
    median_val = np.clip(median_val, 0, None)

    df = pd.DataFrame({'end_time': times, 'median_val': median_val, 'status': 'OK'})
    df.loc[30:35, 'median_val'] = "Error_Text" # 문자열 노이즈 묘사
    return df

sel_df = generate_dummy_data()

# ==========================================
# 1. Data Preprocessing (데이터 전처리)
# ==========================================
# 1-1. 분석에 사용할 컬럼만 추출 및 강제 숫자형 변환 (문자열 등 에러 방지)
df = sel_df.set_index('end_time')[['median_val']].copy()
df['median_val'] = pd.to_numeric(df['median_val'], errors='coerce')

# 1-2. 불규칙 주기 해결: 1시간 단위 정규 리샘플링 후 선형 보간(Interpolation)
df = df.resample('1h').mean()
df['median_val'] = df['median_val'].interpolate(method='linear').bfill().ffill()

# 1-3. 이상치 완화를 위한 Moving Median (Window=5)
df['median_clean'] = df['median_val'].rolling(window=5, center=True).median()
df['median_clean'] = df['median_clean'].bfill().ffill()

# 1-4. Butterworth Low Pass Filter (노이즈/웨이브 제거)
def apply_lowpass_filter(data, cutoff_freq, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff_freq / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# 샘플링 주파수 fs=1(1시간에 1번), Cutoff: 24시간 주기의 변동 제거 (1/24)
df['smoothed'] = apply_lowpass_filter(df['median_clean'], cutoff_freq=1/24, fs=1)

# 1-5. 6시간 단위 리샘플링 (안전을 위해 최대값 Max 사용)
df_6h = df.resample('6h').max().dropna()

# ==========================================
# 2. Degradation Onset Detection (정교한 열화 탐지)
# ==========================================
# 방식: 단순히 임계값을 넘는 것이 아니라, "최근 18시간 동안의 기울기가 뚜렷한 양수" 인지 확인
window_size = 3 # 6h * 3 = 18 hours
df_6h['gradient'] = df_6h['smoothed'].diff(window_size)

threshold_grad = 0.05 # 18시간 동안 0.05 이상 증가 (현업 상황에 맞춰 조정 필요)
consecutive_rises = 2
onset_time = None
count = 0

for idx, row in df_6h.iterrows():
    if pd.notna(row['gradient']) and row['gradient'] > threshold_grad:
        count += 1
        if count >= consecutive_rises:
            # 뚜렷한 상승이 시작된 지점을 Onset으로 지정 (역추적)
            potential_onset = idx - pd.Timedelta(hours=6 * (window_size + consecutive_rises - 1))
            # index 내 존재하는 가장 가까운 시간 찾기
            onset_candidates = df_6h.index[df_6h.index <= potential_onset]
            onset_time = onset_candidates[-1] if len(onset_candidates) > 0 else idx
            break
    else:
        count = 0

# ==========================================
# 3. Model Fitting & RUL Prediction (예측 모델링)
# ==========================================
failure_limit = 1.5
current_time = df_6h.index[-1]
predicted_failure_time = None

# 불확실성 밴드용 변수 초기화
conf_lower_time, conf_upper_time = None, None
curve_lower, curve_upper = None, None
popt, future_times, future_x, future_y = None, None, None, None

if onset_time is None:
    print("No degradation onset detected. (Status: Normal)")
else:
    # Onset 이후의 데이터 추출 및 결측/무한대 값 방어
    fit_data = df_6h[df_6h.index >= onset_time].copy()
    fit_data = fit_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['smoothed'])

    x_data = (fit_data.index - onset_time).total_seconds() / 3600.0
    y_data = fit_data['smoothed'].values

    # 다시 한번 inf/nan 필터링 안전장치
    valid_mask = np.isfinite(x_data) & np.isfinite(y_data)
    x_data = x_data[valid_mask]
    y_data = y_data[valid_mask]

    if len(x_data) < 3:
        print("Not enough valid data points for curve fitting.")
    else:
        # [알고리즘 명시]
        # 1. 기반 예측 모델: Levenberg-Marquardt 최적화를 이용한 비선형 지수 함수 곡선 적합(Non-linear Exponential Curve Fitting)
        def exp_func(x, a, b, c):
            val = a * np.exp(np.clip(b * x, -100, 100)) + c
            return np.clip(val, -1e10, 1e10)

        try:
            initial_guess = (0.01, 0.001, np.min(y_data))
            # popt: 최적 파라미터, pcov: 파라미터 공분산 행렬 (오차 정보)
            popt, pcov = curve_fit(
                exp_func, x_data, y_data,
                p0=initial_guess,
                bounds=([1e-8, 1e-8, -np.inf], [50.0, 1.0, 10.0]),
                maxfev=10000
            )
            a, b, c = popt

            if failure_limit > c:
                t_failure_hours = np.log((failure_limit - c) / a) / b
                predicted_failure_time = onset_time + pd.Timedelta(hours=t_failure_hours)
                rul = predicted_failure_time - current_time

                # 미래 X축 시간 생성 (현재부터 고장시점 이후 +5일 여유)
                future_times = pd.date_range(start=onset_time, end=predicted_failure_time + pd.Timedelta(days=5), freq='6h')
                future_x = (future_times - onset_time).total_seconds() / 3600.0
                future_y = exp_func(future_x, *popt)

                # [알고리즘 명시]
                # 2. 확률적 추정 (Uncertainty Estimation): 파라미터 공분산(pcov) 기반 몬테카를로 시뮬레이션(Monte Carlo Simulation)
                simulated_curves = []

                if not np.isinf(pcov).any():
                    n_simulations = 1000
                    # 다변량 정규분포에서 파라미터 무작위 샘플링
                    sim_params = np.random.multivariate_normal(popt, pcov, n_simulations)
                    sim_failure_hours = []

                    for p in sim_params:
                        a_s, b_s, c_s = p
                        simulated_curves.append(exp_func(future_x, a_s, b_s, c_s))
                        # 각 시뮬레이션 별 고장 도달 시간 계산
                        if a_s > 0 and b_s > 0 and failure_limit > c_s:
                            t_fail = np.log((failure_limit - c_s) / a_s) / b_s
                            sim_failure_hours.append(t_fail)

                    if sim_failure_hours:
                        # 95% 신뢰 구간 (하위 2.5%, 상위 97.5%)
                        ci_lower_hours = np.percentile(sim_failure_hours, 2.5)
                        ci_upper_hours = np.percentile(sim_failure_hours, 97.5)
                        conf_lower_time = onset_time + pd.Timedelta(hours=ci_lower_hours)
                        conf_upper_time = onset_time + pd.Timedelta(hours=ci_upper_hours)

                        simulated_curves = np.array(simulated_curves)
                        curve_lower = np.percentile(simulated_curves, 2.5, axis=0)
                        curve_upper = np.percentile(simulated_curves, 97.5, axis=0)

                print(f"==================================================")
                print(f"       RUL Prediction Results (Probabilistic)     ")
                print(f"==================================================")
                print(f"Current Time:      {current_time.strftime('%Y-%m-%d %H:%M')}")
                print(f"Predicted Failure: {predicted_failure_time.strftime('%Y-%m-%d %H:%M')} (Median Expectation)")
                print(f"RUL (Expected):    {rul.total_seconds() / 3600:.1f} hours ({rul.days} days {rul.components.hours} hours)")

                if conf_lower_time and conf_upper_time:
                    rul_min = (conf_lower_time - current_time).total_seconds() / 3600
                    rul_max = (conf_upper_time - current_time).total_seconds() / 3600
                    print(f"--------------------------------------------------")
                    print(f"95% CI for Failure:{conf_lower_time.strftime('%m-%d %H:%M')} ~ {conf_upper_time.strftime('%m-%d %H:%M')}")
                    print(f"95% CI for RUL:    {rul_min:.1f} hours ~ {rul_max:.1f} hours")
                print(f"==================================================\n")
            else:
                print("Cannot reach failure limit due to flat prediction curve.")

        except Exception as e:
            print(f"Curve Fitting Failed: {e}")

# ==========================================
# 4. Visualization with Plotly (인터랙티브 차트 생성)
# ==========================================
fig = go.Figure()

# 1. Original Data (1H 보간본)
fig.add_trace(go.Scatter(x=df.index, y=df['median_val'], mode='lines', name='Original Sensor Data (1H Interpolated)', line=dict(color='lightgray', width=1), opacity=0.6))

# 2. Smoothed & Resampled Data (6H Max)
fig.add_trace(go.Scatter(x=df_6h.index, y=df_6h['smoothed'], mode='lines+markers', name='Smoothed & Resampled (6H Max)', line=dict(color='blue', width=2), marker=dict(size=5)))

# 3. Upper Limit
fig.add_hline(y=failure_limit, line_dash="dash", line_color="red", annotation_text=f"Failure Limit ({failure_limit})", annotation_position="top left", annotation_font=dict(color="red", size=13))

if onset_time is not None:
    # 4. Onset Point
    fig.add_vline(x=onset_time, line_dash="dot", line_color="orange", annotation_text="Degradation Onset", annotation_position="top right", annotation_font=dict(color="orange"))
    fig.add_trace(go.Scatter(x=[onset_time], y=[df_6h.loc[onset_time, 'smoothed']], mode='markers', name='Onset Point', marker=dict(color='green', size=12, symbol='star')))

    if predicted_failure_time is not None and future_times is not None:
        # 5. Uncertainty Shaded Area (95% CI Band)
        if conf_lower_time is not None:
            # fill='tonexty'를 위해 upper를 먼저 그리고 lower를 나중에 그림 (투명하게)
            fig.add_trace(go.Scatter(x=future_times, y=curve_upper, mode='lines', line=dict(width=0), showlegend=False, name='Upper Bound', hoverinfo='skip'))
            fig.add_trace(go.Scatter(
                x=future_times, y=curve_lower, mode='lines', fill='tonexty', fillcolor='rgba(255, 0, 0, 0.15)',
                line=dict(width=0), name='95% Confidence Interval (Monte Carlo)', hoverinfo='skip'
            ))

        # 6. Extrapolation Line (Current time 기준으로 분리)
        pred_mask = future_times >= current_time
        fit_mask = future_times <= current_time

        fig.add_trace(go.Scatter(x=future_times[fit_mask], y=future_y[fit_mask], mode='lines', name='Fitted Trend (Past)', line=dict(color='green', width=3)))
        fig.add_trace(go.Scatter(x=future_times[pred_mask], y=future_y[pred_mask], mode='lines', name='Predicted Extrapolation (Future)', line=dict(color='red', width=3, dash='dash')))

        # 7. Expected Failure Point Marker
        fig.add_trace(go.Scatter(x=[predicted_failure_time], y=[failure_limit], mode='markers', name='Expected Failure Point (Median)', marker=dict(color='red', size=14, symbol='x')))

# 8. Current Time Line
fig.add_vline(x=current_time, line_dash="dash", line_color="black", annotation_text="Current Time", annotation_position="bottom right")

# Layout Updates
fig.update_layout(
    title=dict(text="Megpie RUL Prediction using Non-linear Curve Fitting & Monte Carlo Simulation", font=dict(size=20, color='darkblue')),
    xaxis_title="Time",
    yaxis_title="Sensor Value (Median)",
    yaxis=dict(range=[0, 1.8]),
    hovermode="x unified",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="lightgray", borderwidth=1),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()